# Bike Temporal Few-Shot Splits

Create chronological few-shot split parquet files for the Seoul bike dataset.

Default interpretation:
- `group = exact date`, so each day is one hourly function.
- `seed0..seed4` encode chronological training-history settings and seed the future validation/test split.
- The temporal split schedule is explicit in `SPLIT_SPECS` below.
- Validation and test are random disjoint groups (days) sampled from the future calendar months.
- Training history increases monotonically across seeds.
- Earlier seeds may use only a part of the whole dataset; `seed4` uses all available dates.
- Support/query points inside validation and test dates are random by default.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from src.constants import (
    CONTEXT_ROLE_COLUMN_NAME,
    GROUPING_COLUMN_NAME,
    RESPONSE_COLUMN_NAME,
    SPLIT_COLUMN_NAME,
    SUPPORT_ROLE_NAME,
    TARGET_ROLE_NAME,
    TEST_SPLIT_NAME,
    TRAIN_SPLIT_NAME,
    VALIDATION_SPLIT_NAME,
)
from src.data.data_layout import PROJECT_ROOT

RAW_SOURCE_PATH = PROJECT_ROOT / "data" / "source" / "SeoulBikeData.csv"
OUT_DIR = PROJECT_ROOT / "data" / "split" / "real" / "bike" / "log_count" / "few_shot"

N_SUPPORT_VALIDATION = 7
N_SUPPORT_TEST = 7

# Options: "random" or "first_hours".
SUPPORT_MODE = "random"

# Explicit chronological split schedule: seed -> (train_months, future_months).
# Validation and test are random disjoint day groups from the future months.
# Validation gets one more day group when the number of future day groups is odd.
# Training history increases monotonically, and seed4 spans all 12 available months.
SPLIT_SPECS = {
    0: (2, 2),
    1: (3, 2),
    2: (4, 4),
    3: (6, 4),
    4: (8, 4),
}

RAW_SOURCE_PATH, OUT_DIR

## Load And Preprocess

The raw cached CSV keeps the true `Date`; the standard `BikeDataProvider` drops it after creating `date_id`, so we preprocess directly here.

In [2]:
def load_bike_source(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run the existing bike raw-data generation first, "
            "or fetch the SeoulBikeData.csv source copy."
        )

    df = pd.read_csv(path)
    df = df[df["Functioning Day"] == "Yes"].copy()
    df["date"] = pd.to_datetime(df["Date"], dayfirst=True)
    df = df.sort_values(["date", "Hour"]).reset_index(drop=True)

    first_date = df["date"].min()
    df["date_id"] = (df["date"] - first_date).dt.days.astype(int)
    df["month_id"] = df["date"].dt.to_period("M")
    df["day_of_week"] = df["date"].dt.dayofweek.astype(int)
    df["day_of_year"] = df["date"].dt.dayofyear.astype(int)

    hour = df["Hour"].astype(float)
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["day_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["day_of_year_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["day_of_year_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    df["log_count"] = np.log1p(df["Rented Bike Count"])

    out = pd.DataFrame(
        {
            "date": df["date"],
            "month_id": df["month_id"].astype(str),
            GROUPING_COLUMN_NAME: df["date_id"],
            "hour": df["Hour"].astype(float),
            "hour_sin": df["hour_sin"],
            "hour_cos": df["hour_cos"],
            "day_of_week_sin": df["day_of_week_sin"],
            "day_of_week_cos": df["day_of_week_cos"],
            "day_of_year_sin": df["day_of_year_sin"],
            "day_of_year_cos": df["day_of_year_cos"],
            "temperature": df["Temperature"].astype(float),
            "humidity": df["Humidity"].astype(float),
            "rainfall": df["Rainfall"].astype(float),
            RESPONSE_COLUMN_NAME: df["log_count"],
        }
    )
    return out


raw_df = load_bike_source(RAW_SOURCE_PATH)
display(raw_df.head())
display(raw_df.groupby("month_id").size().rename("n_rows"))
raw_df.shape, raw_df[GROUPING_COLUMN_NAME].nunique()

,date,month_id,group,hour,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos,temperature,humidity,rainfall,response
0,2017-12-01,2017-12,0,0.0,0.000000,1.000000,-0.433884,-0.900969,-0.497204,0.867634,-5.2,37.0,0.0,5.541264
1,2017-12-01,2017-12,0,1.0,0.258819,0.965926,-0.433884,-0.900969,-0.497204,0.867634,-5.5,38.0,0.0,5.323010
2,2017-12-01,2017-12,0,2.0,0.500000,0.866025,-0.433884,-0.900969,-0.497204,0.867634,-6.0,39.0,0.0,5.159055
3,2017-12-01,2017-12,0,3.0,0.707107,0.707107,-0.433884,-0.900969,-0.497204,0.867634,-6.2,40.0,0.0,4.682131
4,2017-12-01,2017-12,0,4.0,0.866025,0.500000,-0.433884,-0.900969,-0.497204,0.867634,-6.0,36.0,0.0,4.369448


month_id
2017-12    744
2018-01    744
2018-02    672
2018-03    744
2018-04    696
2018-05    720
2018-06    720
2018-07    744
2018-08    744
2018-09    624
2018-10    665
2018-11    648
Name: n_rows, dtype: int64

((8465, 14), 353)

## Split Helpers

Numeric transforms are fit on training rows only, then applied to validation/test. This avoids leaking future distribution information through standardization.

In [3]:
def add_few_shot_roles(
    df: pd.DataFrame,
    split_name: str,
    n_support: int,
    rng: np.random.Generator,
    support_mode: str,
) -> pd.DataFrame:
    df = df.copy()
    df[SPLIT_COLUMN_NAME] = split_name
    df[CONTEXT_ROLE_COLUMN_NAME] = TARGET_ROLE_NAME

    for _, group_df in df.groupby(GROUPING_COLUMN_NAME, sort=False):
        indices = group_df.index.to_numpy()
        k = min(n_support, len(indices) - 1)
        if k <= 0:
            continue

        if support_mode == "random":
            support_indices = rng.choice(indices, size=k, replace=False)
        elif support_mode == "first_hours":
            support_indices = group_df.sort_values("hour").index.to_numpy()[:k]
        else:
            raise ValueError(f"Unknown support_mode: {support_mode}")

        df.loc[support_indices, CONTEXT_ROLE_COLUMN_NAME] = SUPPORT_ROLE_NAME

    return df


def fit_train_only_transforms(train_df: pd.DataFrame, full_df: pd.DataFrame) -> pd.DataFrame:
    full_df = full_df.copy()
    numeric_cols = ["temperature", "humidity", "rainfall"]

    for col in numeric_cols:
        mean = train_df[col].mean()
        std = train_df[col].std(ddof=0)
        if not np.isfinite(std) or std == 0:
            std = 1.0
        full_df[col] = (full_df[col] - mean) / std


    return full_df.drop(columns=["date", "month_id", "hour"])


def split_future_groups(
    future_df: pd.DataFrame,
    rng: np.random.Generator,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    future_groups = future_df[GROUPING_COLUMN_NAME].drop_duplicates().to_numpy()
    future_groups = rng.permutation(future_groups)
    n_validation_groups = int(np.ceil(len(future_groups) / 2))

    validation_groups = future_groups[:n_validation_groups]
    test_groups = future_groups[n_validation_groups:]

    validation = future_df[future_df[GROUPING_COLUMN_NAME].isin(validation_groups)].copy()
    test = future_df[future_df[GROUPING_COLUMN_NAME].isin(test_groups)].copy()
    return validation, test


def build_temporal_few_shot_split(
    df: pd.DataFrame,
    seed: int,
    n_train_months: int,
    n_future_months: int,
    n_support_validation: int,
    n_support_test: int,
    support_mode: str = "random",
) -> pd.DataFrame:
    months = sorted(df["month_id"].unique())
    total_months = n_train_months + n_future_months
    if total_months > len(months):
        raise ValueError(
            f"Need at least {total_months} months for "
            f"train={n_train_months}, future={n_future_months}, "
            f"but only found {len(months)}."
        )

    train_months = months[:n_train_months]
    future_months = months[n_train_months:total_months]

    train = df[df["month_id"].isin(train_months)].copy()
    future = df[df["month_id"].isin(future_months)].copy()

    train[SPLIT_COLUMN_NAME] = TRAIN_SPLIT_NAME
    train[CONTEXT_ROLE_COLUMN_NAME] = None

    rng = np.random.default_rng(seed)
    validation, test = split_future_groups(future, rng)
    validation = add_few_shot_roles(
        validation, VALIDATION_SPLIT_NAME, n_support_validation, rng, support_mode
    )
    test = add_few_shot_roles(test, TEST_SPLIT_NAME, n_support_test, rng, support_mode)

    split_df = pd.concat([train, validation, test], ignore_index=True)
    split_df = fit_train_only_transforms(train, split_df)
    return split_df


def summarize_split(split_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for split_name, sub in split_df.groupby(SPLIT_COLUMN_NAME):
        row = {
            "split": split_name,
            "rows": len(sub),
            "dates": sub[GROUPING_COLUMN_NAME].nunique(),
            "min_group_size": sub.groupby(GROUPING_COLUMN_NAME).size().min(),
            "max_group_size": sub.groupby(GROUPING_COLUMN_NAME).size().max(),
        }
        if CONTEXT_ROLE_COLUMN_NAME in sub.columns:
            row.update(sub[CONTEXT_ROLE_COLUMN_NAME].value_counts(dropna=False).to_dict())
        rows.append(row)
    return pd.DataFrame(rows).sort_values("split")

## Preview The Five Temporal Seeds

In [4]:
preview_rows = []
months = sorted(raw_df["month_id"].unique())
for seed, (n_train_months, n_future_months) in SPLIT_SPECS.items():
    split_df = build_temporal_few_shot_split(
        raw_df,
        seed=seed,
        n_train_months=n_train_months,
        n_future_months=n_future_months,
        n_support_validation=N_SUPPORT_VALIDATION,
        n_support_test=N_SUPPORT_TEST,
        support_mode=SUPPORT_MODE,
    )
    preview_rows.append(
        {
            "seed_file": f"seed{seed}.parquet",
            "train_months": months[:n_train_months],
            "future_months": months[n_train_months : n_train_months + n_future_months],
            "validation_tasks": split_df.loc[split_df[SPLIT_COLUMN_NAME] == VALIDATION_SPLIT_NAME, GROUPING_COLUMN_NAME].nunique(),
            "test_tasks": split_df.loc[split_df[SPLIT_COLUMN_NAME] == TEST_SPLIT_NAME, GROUPING_COLUMN_NAME].nunique(),
            "rows": len(split_df),
            "dates": split_df[GROUPING_COLUMN_NAME].nunique(),
        }
    )

display(pd.DataFrame(preview_rows))
display(summarize_split(split_df))

,seed_file,train_months,future_months,validation_tasks,test_tasks,rows,dates
0,seed0.parquet,"[2017-12, 2018-01]","[2018-02, 2018-03]",30,29,2904,121
1,seed1.parquet,"[2017-12, 2018-01, 2018-02]","[2018-03, 2018-04]",30,30,3600,150
2,seed2.parquet,"[2017-12, 2018-01, 2018-02, 2018-03]","[2018-04, 2018-05, 2018-06, 2018-07]",60,60,5784,241
3,seed3.parquet,"[2017-12, 2018-01, 2018-02, 2018-03, 2018-04, ...","[2018-06, 2018-07, 2018-08, 2018-09]",59,59,7152,298
4,seed4.parquet,"[2017-12, 2018-01, 2018-02, 2018-03, 2018-04, ...","[2018-08, 2018-09, 2018-10, 2018-11]",56,56,8465,353


,split,rows,dates,min_group_size,max_group_size,TARGET,SUPPORT,None
0,TEST,1337,56,17,24,945.0,392.0,NaN
1,TRAIN,5784,241,24,24,NaN,NaN,5784.0
2,VALIDATION,1344,56,24,24,952.0,392.0,NaN


## Write Parquet Files

This cell overwrites only `data/split/real/bike/log_count/few_shot/seed*.parquet`. It leaves the in-context split files untouched.

In [ ]:
WRITE_FILES = True

if WRITE_FILES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    written = []
    for seed, (n_train_months, n_future_months) in SPLIT_SPECS.items():
        split_df = build_temporal_few_shot_split(
            raw_df,
            seed=seed,
            n_train_months=n_train_months,
            n_future_months=n_future_months,
            n_support_validation=N_SUPPORT_VALIDATION,
            n_support_test=N_SUPPORT_TEST,
            support_mode=SUPPORT_MODE,
        )
        out_path = OUT_DIR / f"seed{seed}.parquet"
        split_df.to_parquet(out_path, index=False)
        written.append({"path": str(out_path), "rows": len(split_df)})
    display(pd.DataFrame(written))
else:
    print("Set WRITE_FILES = True to write the temporal few-shot parquet files.")

## Sanity Check A Written File

In [6]:
check_path = OUT_DIR / "seed0.parquet"
if check_path.exists():
    check_df = pd.read_parquet(check_path)
    display(check_df.head())
    display(summarize_split(check_df))
    display(check_df.dtypes)
else:
    print(f"No written file found yet: {check_path}")

,group,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,day_of_year_sin,day_of_year_cos,temperature,humidity,rainfall,response,split,context_role
0,0,0.000000,1.000000,-0.433884,-0.900969,-0.497204,0.867634,-0.412548,-0.777685,-0.083437,5.541264,TRAIN,None
1,0,0.258819,0.965926,-0.433884,-0.900969,-0.497204,0.867634,-0.467213,-0.725678,-0.083437,5.323010,TRAIN,None
2,0,0.500000,0.866025,-0.433884,-0.900969,-0.497204,0.867634,-0.558321,-0.673672,-0.083437,5.159055,TRAIN,None
3,0,0.707107,0.707107,-0.433884,-0.900969,-0.497204,0.867634,-0.594765,-0.621666,-0.083437,4.682131,TRAIN,None
4,0,0.866025,0.500000,-0.433884,-0.900969,-0.497204,0.867634,-0.558321,-0.829691,-0.083437,4.369448,TRAIN,None


,split,rows,dates,min_group_size,max_group_size,TARGET,SUPPORT,None
0,TEST,696,29,24,24,493.0,203.0,NaN
1,TRAIN,1488,62,24,24,NaN,NaN,1488.0
2,VALIDATION,720,30,24,24,510.0,210.0,NaN


group                int64
hour_sin           float64
hour_cos           float64
day_of_week_sin    float64
day_of_week_cos    float64
day_of_year_sin    float64
day_of_year_cos    float64
temperature        float64
humidity           float64
rainfall           float64
response           float64
split               object
context_role        object
dtype: object